In [ ]:
import sys
import os
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(parent_dir)
import tensorflow as tf
from sklearn import preprocessing
import numpy as np
import random
import matplotlib.pyplot as plt
import sys
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset
from matplotlib import cm  
from matplotlib.patches import ConnectionPatch
import pandas as pd
from Model.CNN import cnn_classifier
from utils.LoadData import load_CW_Source,load_CW_Target
from  numba  import njit, prange
from sklearn.utils import shuffle
from Pilsung import*

In [ ]:
@njit
def plot_guessing_entropy_RK(preds, rk0,rk1,Sbox_V, plaintext,trace_num_max):

    """
    - preds : the probability for each class
    - real_key : the key of the target device
    - device_id : id of the target device
    - model_flag : a string for naming GE result
    """
    num_averaged = 20  # Repeat the attack several times to average GE and SR
    
    guessing_entropy = np.zeros((num_averaged, trace_num_max))  # Store GE values for each trace count
    success_flag = np.zeros((num_averaged, trace_num_max))      # Record whether the correct key is ranked first
    
    real_index = rk0 * 256 + rk1  # Convert the true key pair (rk0, rk1) into a flattened index
    
    # Perform multiple attack runs
    for time in range(num_averaged):

        # Randomly select traces for the current attack
        random_index_total = list(range(plaintext.shape[0]))
        random_index_total = np.random.permutation(np.arange(plaintext.shape[0]))
        random_index = random_index_total[0:trace_num_max]

        score_mat = np.zeros((trace_num_max, 256,256))  # Likelihood score for each trace and key hypothesis

        for key_guess0 in range(0, 256):
            for key_guess1 in range(0, 256):

                for i in range(0, trace_num_max):

                    # Compute the intermediate state under the key hypothesis
                    initialState = plaintext[random_index[i]] ^ key_guess0

                    label = np.int64(Sbox_V[key_guess1,initialState])

                    # Use the model prediction probability as the likelihood score
                    score_mat[i, key_guess0,key_guess1] = preds[random_index[i], label]

        cumulative_score = np.zeros_like(score_mat)
        cumulative_score[0,:,:] = score_mat[0,:,:]

        # Accumulate the likelihood scores across traces
        for i in range(1, trace_num_max):
            cumulative_score[i,:,:] = cumulative_score[i-1,:,:] + score_mat[i,:,:]

        for i in range(trace_num_max):

            # Obtain the cumulative log-likelihood for all key hypotheses
            log_likelihood = cumulative_score[i]

            # Flatten the key space (256×256) for ranking
            flat = log_likelihood.reshape(-1)

            # Rank all key hypotheses according to likelihood
            ranked = np.argsort(flat)[::-1]

            # Compute the rank of the correct key
            rank = np.where(ranked == real_index)[0][0]

            guessing_entropy[time, i] = rank

            # Check if the correct key is ranked first
            if ranked[0] == real_index:
                success_flag[time, i] = 1
 
    guessing_entropy_avg = np.zeros(trace_num_max)
    for i in range(trace_num_max):
        guessing_entropy_avg[i] = np.sum(guessing_entropy[:, i]) / num_averaged

    success_flag = np.sum(success_flag, axis=0)
    success_rate = success_flag/num_averaged 
    
    return guessing_entropy_avg, success_rate

In [ ]:
# Main parameter initialization
profiling_Data_path='../DataSet/rk1=0x35/'
Target_Data_path = '../DataSet/rk1=0x2b/'
model_path = '../Model/'
model_name = 'Source_Model(RK1).h5'

In [ ]:
prediction_byte=[]
p_total=[]
byte=0

# Load profiling traces (template attack dataset)
profiling_traces, _, _, _, _, _ = load_CW_Source(
        in_file=profiling_Data_path,
        sec=18000,  # Fixed security parameter from original implementation
)

    # Load target device data
X_attack, label_V, p_attack = load_CW_Target(
        in_file=Target_Data_path,
    )

# Preprocessing pipeline
# 1. Standardization (zero-mean, unit-variance)
scaler = preprocessing.StandardScaler()
profiling_traces = scaler.fit_transform(profiling_traces)
X_attack = scaler.transform(X_attack)

# # 2. Normalization (scale to [0,1] range)
scaler = preprocessing.MinMaxScaler(feature_range=(0, 1))
profiling_traces = scaler.fit_transform(profiling_traces)
X_attack = scaler.transform(X_attack)
X_attack = X_attack.reshape(X_attack.shape)

# load model
model = cnn_classifier(input_size=600)
model.load_weights(model_path + model_name)
predictions = model.predict(X_attack)
prediction_byte.append(predictions)
p_total.append(p_attack)

In [5]:
#Cache all possible S-boxes
def generateSbox(k1):
    """Generate S-box for first round key candidate (rk1=k1)"""
    key_total = np.zeros(32, dtype=np.int32)
    key_total[16] = k1  # Set rk1 position
    gen_enc_perm(key_total, current_permutation_8, pboxes)
    return np.array(sboxes)  

# Initialize S-box storage
Sbox_V = np.zeros((256, 256))  # S-box values: [rk1][input_byte]
diffnum = np.zeros((256, 256))  # Differential analysis matrix
v_unique_num = np.zeros(256)  # Unique S-box output counts

#Generate all possible S-boxes
for k1 in range(256):
    Sbox_gen = generateSbox(k1)
    for v in range(256):
        Sbox_V[k1][v] = Sbox_gen[0][0][v]  # Store S-box values

In [ ]:
GE_inf = []  # Initialize list to store guessing entropy results for multiple runs
real_key1 = 0x74  # Set the first byte of the real key
real_key2 = 0x2b  # Set the second byte of the real key
trace_num_max = 5000  # Maximum number of traces to use in the attack
predictions = prediction_byte[byte]  
p_attack = np.int64(p_total[byte])  
# Compute guessing entropy and success rate using the RK attack function
guessing_entropy, success_rate = plot_guessing_entropy_RK(
    predictions, real_key1, real_key2, Sbox_V, p_attack, trace_num_max
)
GE_inf.append(guessing_entropy)  